# ApplySafe Scam Classifier — from-scratch Transformer

Trains a custom Transformer text encoder (no pretrained weights) combined with structured
job-posting metadata to classify postings as fraudulent or legitimate.

Dataset: Kaggle "Real / Fake Job Posting Prediction" (`fake_job_postings.csv`).

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

## 1. Upload the dataset
Run this cell and select `fake_job_postings.csv` when prompted.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/content/fake_job_postings.csv'):
    uploaded = files.upload()
    fname = next(iter(uploaded))
    os.rename(fname, '/content/fake_job_postings.csv')
print('dataset ready at /content/fake_job_postings.csv')

## 2. Preprocessing
Merges text fields, builds a vocab from the train split only (no leakage), label-encodes
categorical fields, and does a stratified 80/10/10 split (the fraud class is only ~4.8% of rows).

In [ ]:
import json, re, os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

DATA_PATH = '/content/fake_job_postings.csv'
ARTIFACTS_DIR = '/content/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TEXT_COLS = ['title', 'company_profile', 'description', 'requirements', 'benefits']
CAT_COLS = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
MAX_LEN = 400
VOCAB_SIZE = 20000
MIN_FREQ = 2
PAD, UNK, CLS = '<pad>', '<unk>', '<cls>'
TOKEN_RE = re.compile(r"[a-z0-9']+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

def build_vocab(texts):
    freq = {}
    for text in texts:
        for tok in tokenize(text):
            freq[tok] = freq.get(tok, 0) + 1
    kept = [tok for tok, count in sorted(freq.items(), key=lambda x: -x[1]) if count >= MIN_FREQ]
    kept = kept[:VOCAB_SIZE - 3]
    vocab = {PAD: 0, UNK: 1, CLS: 2}
    for tok in kept:
        vocab[tok] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len=MAX_LEN):
    ids = [vocab[CLS]] + [vocab.get(tok, vocab[UNK]) for tok in tokenize(text)]
    ids = ids[:max_len]
    attn = [1] * len(ids)
    pad_len = max_len - len(ids)
    ids += [vocab[PAD]] * pad_len
    attn += [0] * pad_len
    return ids, attn

df = pd.read_csv(DATA_PATH)
df['full_text'] = df[TEXT_COLS].fillna('').agg(' '.join, axis=1)
df['has_salary'] = df['salary_range'].notna().astype(int)
for col in CAT_COLS:
    df[col] = df[col].fillna('__missing__')

labels = df['fraudulent'].values
idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=labels[temp_idx], random_state=42)
print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')

vocab = build_vocab(df.iloc[train_idx]['full_text'])
print(f'vocab size: {len(vocab)}')

cat_maps = {}
for col in CAT_COLS:
    cats = sorted(df.iloc[train_idx][col].unique())
    cat_maps[col] = {'__unk__': 0, **{c: i + 1 for i, c in enumerate(cats)}}

with open(f'{ARTIFACTS_DIR}/vocab.json', 'w') as f:
    json.dump(vocab, f)
with open(f'{ARTIFACTS_DIR}/cat_maps.json', 'w') as f:
    json.dump(cat_maps, f)
with open(f'{ARTIFACTS_DIR}/config.json', 'w') as f:
    json.dump({'max_len': MAX_LEN, 'cat_cols': CAT_COLS}, f)

def build_split(split_idx, name):
    sub = df.iloc[split_idx]
    input_ids = torch.zeros((len(sub), MAX_LEN), dtype=torch.long)
    attn_mask = torch.zeros((len(sub), MAX_LEN), dtype=torch.long)
    for i, text in enumerate(sub['full_text']):
        ids, attn = encode_text(text, vocab)
        input_ids[i] = torch.tensor(ids)
        attn_mask[i] = torch.tensor(attn)

    cat_ids = torch.zeros((len(sub), len(CAT_COLS)), dtype=torch.long)
    for j, col in enumerate(CAT_COLS):
        cat_ids[:, j] = torch.tensor([cat_maps[col].get(v, 0) for v in sub[col]], dtype=torch.long)

    bin_features = torch.tensor(
        sub[['telecommuting', 'has_company_logo', 'has_questions', 'has_salary']].values,
        dtype=torch.float,
    )
    y = torch.tensor(sub['fraudulent'].values, dtype=torch.float)

    torch.save(
        {'input_ids': input_ids, 'attn_mask': attn_mask, 'cat_ids': cat_ids, 'bin_features': bin_features, 'labels': y},
        f'{ARTIFACTS_DIR}/{name}.pt',
    )
    print(f'saved {name}.pt: {len(sub)} rows, {int(y.sum())} positive')

build_split(train_idx, 'train')
build_split(val_idx, 'val')
build_split(test_idx, 'test')

## 3. Model
Token + learned positional embeddings feed a Transformer encoder (randomly initialized,
no pretrained checkpoint). The `[CLS]` token's output is pooled as the text representation
and concatenated with embedded categorical/binary posting metadata before the classification head.

In [ ]:
import torch.nn as nn

class ScamClassifier(nn.Module):
    def __init__(self, vocab_size, max_len, cat_cardinalities, num_bin_features,
                 d_model=128, nhead=4, num_layers=4, dim_feedforward=256, cat_emb_dim=16, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.text_dropout = nn.Dropout(dropout)

        self.cat_embs = nn.ModuleList([nn.Embedding(card, cat_emb_dim, padding_idx=0) for card in cat_cardinalities])
        struct_dim = cat_emb_dim * len(cat_cardinalities) + num_bin_features

        self.head = nn.Sequential(
            nn.Linear(d_model + struct_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attn_mask, cat_ids, bin_features):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(positions)

        padding_mask = attn_mask == 0
        x = self.encoder(x, src_key_padding_mask=padding_mask)
        pooled_text = self.text_dropout(x[:, 0, :])

        cat_vecs = [emb(cat_ids[:, i]) for i, emb in enumerate(self.cat_embs)]
        struct = torch.cat(cat_vecs + [bin_features], dim=1)

        combined = torch.cat([pooled_text, struct], dim=1)
        return self.head(combined).squeeze(-1)

## 4. Training
Uses a class-weighted `BCEWithLogitsLoss` since only ~4.8% of postings are fraudulent —
accuracy alone would be misleading, so precision/recall/F1/AUC are tracked instead.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
PATIENCE = 3

def load_split(name):
    d = torch.load(f'{ARTIFACTS_DIR}/{name}.pt')
    return TensorDataset(d['input_ids'], d['attn_mask'], d['cat_ids'], d['bin_features'], d['labels'])

def make_loader(name, shuffle):
    return DataLoader(load_split(name), batch_size=BATCH_SIZE, shuffle=shuffle)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for input_ids, attn_mask, cat_ids, bin_features, labels in loader:
            input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
            cat_ids, bin_features, labels = cat_ids.to(DEVICE), bin_features.to(DEVICE), labels.to(DEVICE)
            logits = model(input_ids, attn_mask, cat_ids, bin_features)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, preds, average='binary', zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader.dataset), precision, recall, f1, auc

with open(f'{ARTIFACTS_DIR}/cat_maps.json') as f:
    cat_maps = json.load(f)
with open(f'{ARTIFACTS_DIR}/config.json') as f:
    config = json.load(f)
cat_cardinalities = [len(cat_maps[col]) for col in config['cat_cols']]

train_loader = make_loader('train', shuffle=True)
val_loader = make_loader('val', shuffle=False)

model = ScamClassifier(
    vocab_size=len(vocab), max_len=config['max_len'],
    cat_cardinalities=cat_cardinalities, num_bin_features=4,
).to(DEVICE)

train_labels = load_split('train').tensors[-1]
n_pos = train_labels.sum().item()
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([n_neg / n_pos]).to(DEVICE)
print(f'pos_weight (class imbalance correction): {pos_weight.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_f1 = 0.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for input_ids, attn_mask, cat_ids, bin_features, labels in train_loader:
        input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)
        cat_ids, bin_features, labels = cat_ids.to(DEVICE), bin_features.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, cat_ids, bin_features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    train_loss = total_loss / len(train_loader.dataset)
    val_loss, precision, recall, f1, auc = evaluate(model, val_loader, criterion)
    print(f'epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} '
          f'precision={precision:.3f} recall={recall:.3f} f1={f1:.3f} auc={auc:.3f}')

    if f1 > best_f1:
        best_f1 = f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), f'{ARTIFACTS_DIR}/best_model.pt')
        print(f'  -> saved new best model (f1={f1:.3f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'early stopping at epoch {epoch}')
            break

print(f'training done. best val f1={best_f1:.3f}')

## 5. Test-set evaluation

In [ ]:
test_loader = make_loader('test', shuffle=False)
model.load_state_dict(torch.load(f'{ARTIFACTS_DIR}/best_model.pt'))
test_loss, precision, recall, f1, auc = evaluate(model, test_loader, criterion)
print(f'TEST: loss={test_loss:.4f} precision={precision:.3f} recall={recall:.3f} f1={f1:.3f} auc={auc:.3f}')

## 6. Download the trained artifacts
Grabs `best_model.pt`, `vocab.json`, `cat_maps.json`, `config.json` as a zip so you can
bring them back into the ApplySafe repo for inference.

In [ ]:
import shutil
from google.colab import files as colab_files

shutil.make_archive('/content/scam_classifier_artifacts', 'zip', ARTIFACTS_DIR)
colab_files.download('/content/scam_classifier_artifacts.zip')